In [1]:
import pandas as pd
from math import floor
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np
import chart_studio
import chart_studio.plotly as py
import plotly.io as pio
pd.set_option('display.max_columns', None)

In [2]:
ELECCIONES_PROVINCIA_FILES_PATH = "../../data/raw/resultados_electorales/congreso_por_provincia/PROV_02_201911_1.xlsx"
ELECCIONES_MUNICIPIO_PATH = "../../data/staging/elecciones_generales_completas.csv"
ESCANIOS_PATH = "../../data/raw/escaños/escaños_2019.xlsx"
GEOJSON_PATH = "../../data/raw/mapa_provincias/mapa_provincias.json"

username = 'politicai' # your username
api_key = 'REDACTED' # your api key - go to profile > settings > regenerate key
chart_studio.tools.set_credentials_file(username=username, api_key=api_key)
# py.plot(fig_object, filename = 'gdp_per_cap', auto_open=True)
# pio.write_html(fig, file=’index.html’, auto_open=True)

In [3]:
coalicion_a_favor = ["psoe", "podemos_iu", "eaj_pnv", "mas_pais_equo", "bng" ,
                     "teruel_existe!", "ecp_guanyem_el_canvi", "podemos_eu", "mes_compromis", "mas_pais", "m_pais_cha_equo"]
coalicion_en_contra = ["pp", "vox", "cs", "jxcat_junts", "cup_pr", "upn", "cca_pnc_nc", "Foro Asturias","prc", "na+"]
coalicion_abstencion = ["erc_sobiranistes", "eh_bildu"]

In [4]:
def clean_column_names(df):
    df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(':', '').str.replace('(', '')
    df.columns = df.columns.str.replace('.', '').str.replace(')', '').str.replace('-', '_').str.replace(')', '')
    df.columns = df.columns.str.replace("á", "a").str.replace("é", "e").str.replace("í", "i").str.replace("ó", "o").str.replace("ú", "u")
    return df

def limpiar_elecciones_municipio(df):
    key_municipio_columns = ['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
               'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
               'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
               'votos_validos', 'votos_a_candidaturas','votos_en_blanco','votos_nulos',
               'year', 'month', 'yearmonth', 'CODIGOINE']
    party_columns_municipio = [x for x in df.columns if x not in key_municipio_columns]
    df[party_columns_municipio] = df[party_columns_municipio].apply(pd.to_numeric)
    return df, party_columns_municipio


def limpiar_elecciones_provincia(df):
    diputados_cols = [x for x in df.columns if "Diputados" in x]
    df = df.drop(columns=diputados_cols)

    for idx, elem in enumerate(df.columns.values):
        if "Unnamed" in elem[0]:
            df.columns.values[idx]=("",df.columns.values[idx][1])
    df.columns = [' '.join(col).strip() for col in df.columns.values]
    df = clean_column_names(df)
    df.columns = [col.replace("_votos", "") for col in df.columns]

    numeric_columns = [x for x in df.columns if x not in ["nombre_de_comunidad", "codigo_de_provincia", "nombre_de_provincia"]]
    df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric)
    key_prov_columns = ["nombre_de_comunidad","codigo_de_provincia","nombre_de_provincia",
               "poblacion","numero_de_mesas","censo_electoral_sin_cera","censo_cera",
               "total_censo_electoral","solicitudes_voto_cera_aceptadas","total_votantes_cer",
               "total_votantes_cera","total_votantes","votos_validos","votos_a_candidaturas",
               "votos_en_blanco","votos_nulos"]
    party_columns_prov = [x for x in df.columns if x not in key_prov_columns]
    
    return df, party_columns_prov

def calculate_dhondt(df_row_votes, party_columns, n_seats, min_per=0.03):
    # matriz base
    parties_T = df_row_votes[party_columns].T
    total_votos = int(parties_T.sum())
    parties_T = parties_T[parties_T[parties_T.columns[0]] > 0.03*total_votos]
    parties_T = parties_T.sort_values(by = parties_T.columns[0], ascending=False) 
    parties_T = parties_T.rename(columns={parties_T.columns[0]:"votos"})
    #escaños acumulados
    current_votes = parties_T.copy()
    current_votes["acumulado"] = 1
    current_votes = current_votes.drop(columns =["votos"])
    #calculo
    for idx in range(1,n_seats+1):
        parties_T[f'escaño_{idx}'] = (parties_T["votos"] / current_votes["acumulado"]).astype(int)
        max_col = parties_T[f'escaño_{idx}'].idxmax()
        #print(f"escaño {idx}: ", max_col)
        current_votes.loc[current_votes.index == max_col,"acumulado"] += 1 
    current_votes["acumulado"] = current_votes["acumulado"] -1
    
    return parties_T.T, current_votes

## Datos electorales municipio

In [72]:
df_elecciones_municipio = pd.read_csv(ELECCIONES_MUNICIPIO_PATH, sep="|", dtype={"CODIGOINE":str}, index_col=0)
df_elecciones_municipio, party_columns_municipio = limpiar_elecciones_municipio(df_elecciones_municipio)
print(df_elecciones_municipio.shape)
df_elecciones_municipio.head(2)

(121361, 645)


,nombre_de_comunidad,codigo_de_provincia,nombre_de_provincia,codigo_de_municipio,nombre_de_municipio,poblacion,numero_de_mesas,total_censo_electoral,total_votantes,votos_validos,votos_a_candidaturas,votos_en_blanco,votos_nulos,pp,psoe,iu,ciu,eaj_pnv,cc,bng,hb,erc,pa,ea,uv,lve,cha,uc,upv_bn,psm_ene,cdn,lv_gv,prt,pcpe,ph,pas,upl,fea,icv_gorordo,lv,cex,um,tc_pnc,priojano,pec,urcl,na,aun,cr,sos,prepal,frepic,pspc,prcm,fpg,siex,prv,prim,fei,nr,picc,ar,env,pae,pb,pcn,upra,uad,adec,vdpa,n,alas,aba,prgu,lae,dsa,lg,piz,pnm,por,afit,ccv,year,month,yearmonth,ciudadanos,podemos_iu_equo,vox,erc_sobiranistes,ecp_guanyem_el_canvi,jxcat_junts,eh_bildu,podemos_eu_mareas_en_comun_equo,cca_pnc,na+,compromis_2019,prc,pacma,front_republica,recortes_cero_gv,nca,pact,ara_mes_esquerra,gbai,pum+j,en_marea,pcte,el_pi,axsi,pcoe,avant_adelante_los_verdes,eb,cpm,somos_region,pcpa,uig_som_cuides,erpv,izqp,pcpc,ahora_canarias,cxg,ppso,cnv,cex_crex_prex,pr+,p_lib,cilu_linares,andecha_astur,"""jf""",pyln,fia,fe_de_las_jons,f8,solidaria,dpl,union_regionalista,centrados,vou,dp,pdsje_udec,izar,risa,c_21,+mas+,udt,ap_pdp,ucd,pce,cds,fn,pst,ee,psa_pa,pcc,b_psg,upc,ne,se,uce,pce_m_l,eu,eg,ac_cc,upv,cuc,ps,mfe,eupv,aeba,psm,lc_cosi,psa,env_urv,aegi,ce_fcc,fe_jons,fic,pnc,padl,cc1,idg,fe_i,lcr_mc,lcr,aci,pre,pnc1,pdb,mcate,asdc,ir,fa,em,pcu,pc,emk,bia,cir,podemos,en_comu,iu_upec,podemos_com,erc_catsi,dl,podemos_en,upyd,nos,uniocat,recortes_ce,mes,ccd,fe_de_las_j,x_la_izquie,somval,sain,centro_mode,pt,canarias_de,arapv,eu_ex,dn,ifem,en_positiu,cilus,cra,ln,avant,mlgxsi,pfyv,aju,independien,fdee,soluciona,"js,pc",pld,oe,ruiz_mateos,lv_lv,pte_uc,par,aic,cg,psg_eg,av_mec,verde,acn,csd,png_pg,exu,pore,igc,ped,pam,grm,una,ua_cha,ub,ceu,arde,sev,mv,pancal,coa,pee_lv,r_x_c,tagoror,prb,pnem,ppr,lki_emk,pce_ml,cd,un,pnv,pte,psoe_h,ort,erfn,mc_oic,bnpg,c_upc,pg_pog_psg,bean,pcarl,pct,ufv,fjonsa,upn,oce_br,psar___psda,plib,urv,pnpv,unai,pdpcan,smim,psin,ule,e_catala,cic,p_cant,dscc,pprov,lc,ocebr_uce,cna,pippa,poc,cime,fe_uf,uf_fi_at,verdes,ea_eue,le,arm_ade,pap,upca,ua,gil,pgc,pln,e_u,ummp,pec_verde,nps,p_riojano,cng,ag,prex,sees,arcoiris,app,pvpa,mce,atf,aigranc,fpe,aa,fe_a,unida,udla,os,conceyu,piap,abe,pitrcg,if,p_tagoror,mces,us,ince,freixes,ccl,ap,psp_us,pdpc,fdc_edc,udc_idcc,ec_fed,fdi,asdci,aet,an18,rse,fut,caic,esb,pspv,indep,dcv,psg,fjons,afn,indep1,ura,lliga,uan,anepa_cp,fl,div,cja,cups,ccia,uab,uras,fni,ppcan,ms,asa,mfa,anv,indep2,indep3,indep4,dcar,udib,indep5,cup,pscan,faa,pim,indep6,prsv,icu,psde,indep7,indep8,pdg,indep9,indep10,indep11,aeicyu,cuir,cipye,pli,adc,bai,indep12,cai,ic_v,bloc_verds,psm_en,uc_cds,lv_cm,fe,posi,ev_av,pil,es2000,pade,um_inme,ia,blm,fei_fe_2000,pcl,autonomo,ec,pkd,el_papi,a,prex_crex,dg,upz,cnc,peda,li_lit_ci,paviel,ir_pre,usi,pnca,inar,up_can,ural,ccs,csd_l,pf,ppcr,upb,ii,unib,prib,tpc,cives,mash,pdep,an,eh,iu_lv,amaiur,esquerra,equo,cc_nc_pnc,compromis_q,fac,pxc,anticapitalistas,piratacat,espana_2000,rps,hartosorg,pirata,anc,cdl,pcas,uxv,prao,caballas,cxxi,unidad_del_pueblo,andecha,acima,ucit,dec,cyd,prde,internet,pre_r,basta_ya,pdyc,hd,sxt,dne,cc_pnc,ca,na_bai,nc_ccn,aralar,bloc_idpv_ev_ee,unitat_per_les_illes,psd,cenb,ev_ae,pdeal,rcn_nok,ev_lv,aes,e_2000,rc,cva,ei,tc,amd,pdlpea,partido_riojano,abla,extremadura_unida,pxcat,pnf_no_fumadores,uleg,onv,frente,cdes,plci,pleve,li_litci,plrv,centristas_pctr,pcua,ucpic,irv,mupc,puede,piib,pb_ub,ctc,ud_ca,imc,paie,td,cdas,ave,ucl,n_som,axb,ixc,icbur,agruci,phache,ap_pdp_pl,muc,prd,ac_inc,av,upr,pspa,cun,psdc,adei,cl,loc,cn,pmcn,prm,prov,ae_aic,paec,lki,podemos_iu,mas_pais_equo,cup_pr,podemos_eu,cca_pnc_nc,mes_compromis,teruel_existe!,mas_pais,m_pais_cha_equo,mes_esquerra,xav,contigo,pdsje,mas,unidos_si_acps_def,mdyc,aunacv,canda,ecp,podemos_compromis_eupv,cdc,podemos_en_marea_anova_eu,recortes_cero_grupo_verde,bng_nos,si,centro_moderado,ccd_ci,and,jxc,repo,independientes_fia,entaban,alcd,fme,hrts_ln,bloc_ev,"psm_en,eu,ev,er",aralar_zutik,lv_e,lvcm,pcannabis,lvrm,msr,ides,apca,gve,ci,adn,ptpre,up

## Datos electorales provincia noviembre 2019


In [6]:
df_provincia_nov_2019_raw = pd.read_excel(io=ELECCIONES_PROVINCIA_FILES_PATH, header=[0,1], skiprows=4)
df_provincia, party_columns_prov = limpiar_elecciones_provincia(df_provincia_nov_2019_raw)
print(df_provincia.shape)
df_provincia.head(2)

(54, 83)


,nombre_de_comunidad,codigo_de_provincia,nombre_de_provincia,poblacion,numero_de_mesas,censo_electoral_sin_cera,censo_cera,total_censo_electoral,solicitudes_voto_cera_aceptadas,total_votantes_cer,total_votantes_cera,total_votantes,votos_validos,votos_a_candidaturas,votos_en_blanco,votos_nulos,psoe,pp,vox,podemos_iu,erc_sobiranistes,cs,jxcat_junts,ecp_guanyem_el_canvi,eaj_pnv,eh_bildu,mas_pais_equo,cup_pr,podemos_eu,cca_pnc_nc,na+,mes_compromis,bng,prc,teruel_existe!,pacma,mas_pais,recortes_cero_gv,pum+j,m_pais_cha_equo,mes_esquerra,axsi,pcpe,pcte,gbai,upl,pcoe,cpm,eb,erpv,xav,avant_adelante_los_verdes,verdes,ph,ifem,uig_som_cuides,somos_region,izqp,ahora_canarias,contigo,cha,ppso,extremadura_unida,pdsje,p_lib,mas,unidos_si_acps_def,prepal,andecha,mdyc,pyln,fe_de_las_jons,aunacv,union_regionalista,canda,fia,solidaria,risa,centrados,dpl,izar,c_21,udt
0,Andalucía,4,Almería,709340.0,809.0,460639.0,41988.0,502627.0,2923.0,303481.0,1933.0,305414.0,302424.0,299763.0,2661.0,2990.0,89295.0,78072.0,80714.0,24400.0,0.0,22835.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2582.0,0.0,360.0,371.0,0.0,0.0,318.0,303.0,0.0,0.0,0.0,0.0,0.0,513.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Andalucía,11,Cádiz,1238714.0,1520.0,973238.0,29057.0,1002295.0,3485.0,621965.0,2230.0,624195.0,616079.0,606858.0,9221.0,8116.0,188271.0,111089.0,131205.0,93541.0,0.0,55490.0,0.0,0.0,0.0,0.0,11316.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8437.0,0.0,915.0,707.0,0.0,0.0,3509.0,639.0,0.0,0.0,0.0,1739.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Escaños

In [8]:
df_escanios = pd.read_excel(ESCANIOS_PATH)
df_escanios.head()

,codigo_de_provincia,nombre_de_provincia,escaños
0,15,A Coruña,8
1,2,Albacete,4
2,3,Alicante / Alacant,12
3,4,Almería,6
4,1,Araba / Álava,4


## Dhondt

In [9]:
prov_full_dict = {}
for index, row in df_escanios.iterrows():
    #print(row["nombre_de_provincia"])
    df_provincia_row = df_provincia[df_provincia.codigo_de_provincia == row["codigo_de_provincia"]]
    tabla_dhondt, escanios = calculate_dhondt(df_row_votes=df_provincia_row, 
                                              party_columns=party_columns_prov, n_seats=row["escaños"])

    prov_full_dict[str(row["codigo_de_provincia"])]={}
    prov_full_dict[str(row["codigo_de_provincia"])]["tabla"] = tabla_dhondt
    prov_full_dict[str(row["codigo_de_provincia"])]["escaños"] = escanios
    #print(tabla_dhondt.tail(2))

In [10]:
prov_diff_dict = {}
df_prov_diff = pd.DataFrame(data=[], columns=["codigo_de_provincia", "nombre_de_provincia", "diferencia"])
for cod_prov in df_escanios["codigo_de_provincia"].unique():
    nombre_prov = df_escanios[df_escanios["codigo_de_provincia"] == cod_prov]["nombre_de_provincia"].to_list()[0]
    print(nombre_prov)
    last_line = prov_full_dict[str(cod_prov)]["tabla"].tail(1)
    gold = last_line.idxmax(axis=1).to_list()
    silver = last_line[[x for x in last_line.columns if x not in gold]].idxmax(axis=1).to_list()
    last_line = last_line[gold+silver]
    prov_plus = [x for x in last_line.columns if x in coalicion_a_favor]
    prov_minus = [x for x in last_line.columns if x in coalicion_en_contra]
    last_line["a_favor"] = last_line[prov_plus].sum(axis=1)
    last_line["en_contra"] = last_line[prov_minus].sum(axis=1)
    last_line["diferencia"] = last_line["a_favor"] - last_line["en_contra"]
    prov_diff_dict[str(cod_prov)] = last_line
    df_prov_diff.loc[cod_prov] = [str(cod_prov).zfill(2), nombre_prov, last_line["diferencia"].to_list()[0]]
    print(last_line)

A Coruña                      
              bng      vox  a_favor  en_contra  diferencia
escaño_8  58135.0  50325.0  58135.0    50325.0      7810.0
Albacete                      
             psoe       pp  a_favor  en_contra  diferencia
escaño_4  34924.0  29417.0  34924.0    29417.0      5507.0
Alicante / Alacant            
               vox  podemos_iu  a_favor  en_contra  diferencia
escaño_12  55798.0     54266.0  54266.0    55798.0     -1532.0
Almería                       
               pp     psoe  a_favor  en_contra  diferencia
escaño_6  39036.0  29765.0  29765.0    39036.0     -9271.0
Araba / Álava                 
          eh_bildu       pp  a_favor  en_contra  diferencia
escaño_4   27453.0  25506.0      0.0    25506.0    -25506.0
Asturias                      
             psoe  podemos_iu   a_favor  en_contra  diferencia
escaño_7  62070.0     44650.0  106720.0        0.0    106720.0
Ávila                         
              vox       pp  a_favor  en_contra  diferenci

In [11]:
df_prov_best_candidates = df_prov_diff[df_prov_diff.diferencia > 0].sort_values(ascending=True, by="diferencia")
df_prov_best_candidates.head()

,codigo_de_provincia,nombre_de_provincia,diferencia
16,16,Cuenca,227.0
22,22,Huesca,1813.0
17,17,Girona,2007.0
12,12,Castellón / Castelló,4163.0
2,02,Albacete,5507.0


# Mapa España por provincias

In [20]:
#cargar
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)

In [50]:
## GRAFICO --------------------
# Viridis_r  Cividis_r
fig_map_frozen = px.choropleth_mapbox(data_frame=df_prov_best_candidates, 
                           geojson=geojson, locations='codigo_de_provincia', 
                           featureidkey="properties.Codigo", color='diferencia', mapbox_style="carto-positron", 
                           hover_data=["nombre_de_provincia","diferencia"], zoom=4, color_continuous_scale="Viridis_r",
                           center={"lat": 35, "lon": -7}, height=800,
                           title= f"Diferencia votos 2019-11")
fig_map_frozen.show()

# Mapa municipios de provincia

# Evolución votos municipio

In [73]:
# evolucion partidos mas votados, lineas dejar rastro, filtro por municipio
CODIGOINE_MUNICIPIO="28092"

In [74]:
df_municipio = df_elecciones_municipio[df_elecciones_municipio.CODIGOINE == CODIGOINE_MUNICIPIO]
nombre_municipio = df_municipio.nombre_de_municipio.unique()[0]
df_municipio.head()

,nombre_de_comunidad,codigo_de_provincia,nombre_de_provincia,codigo_de_municipio,nombre_de_municipio,poblacion,numero_de_mesas,total_censo_electoral,total_votantes,votos_validos,votos_a_candidaturas,votos_en_blanco,votos_nulos,pp,psoe,iu,ciu,eaj_pnv,cc,bng,hb,erc,pa,ea,uv,lve,cha,uc,upv_bn,psm_ene,cdn,lv_gv,prt,pcpe,ph,pas,upl,fea,icv_gorordo,lv,cex,um,tc_pnc,priojano,pec,urcl,na,aun,cr,sos,prepal,frepic,pspc,prcm,fpg,siex,prv,prim,fei,nr,picc,ar,env,pae,pb,pcn,upra,uad,adec,vdpa,n,alas,aba,prgu,lae,dsa,lg,piz,pnm,por,afit,ccv,year,month,yearmonth,ciudadanos,podemos_iu_equo,vox,erc_sobiranistes,ecp_guanyem_el_canvi,jxcat_junts,eh_bildu,podemos_eu_mareas_en_comun_equo,cca_pnc,na+,compromis_2019,prc,pacma,front_republica,recortes_cero_gv,nca,pact,ara_mes_esquerra,gbai,pum+j,en_marea,pcte,el_pi,axsi,pcoe,avant_adelante_los_verdes,eb,cpm,somos_region,pcpa,uig_som_cuides,erpv,izqp,pcpc,ahora_canarias,cxg,ppso,cnv,cex_crex_prex,pr+,p_lib,cilu_linares,andecha_astur,"""jf""",pyln,fia,fe_de_las_jons,f8,solidaria,dpl,union_regionalista,centrados,vou,dp,pdsje_udec,izar,risa,c_21,+mas+,udt,ap_pdp,ucd,pce,cds,fn,pst,ee,psa_pa,pcc,b_psg,upc,ne,se,uce,pce_m_l,eu,eg,ac_cc,upv,cuc,ps,mfe,eupv,aeba,psm,lc_cosi,psa,env_urv,aegi,ce_fcc,fe_jons,fic,pnc,padl,cc1,idg,fe_i,lcr_mc,lcr,aci,pre,pnc1,pdb,mcate,asdc,ir,fa,em,pcu,pc,emk,bia,cir,podemos,en_comu,iu_upec,podemos_com,erc_catsi,dl,podemos_en,upyd,nos,uniocat,recortes_ce,mes,ccd,fe_de_las_j,x_la_izquie,somval,sain,centro_mode,pt,canarias_de,arapv,eu_ex,dn,ifem,en_positiu,cilus,cra,ln,avant,mlgxsi,pfyv,aju,independien,fdee,soluciona,"js,pc",pld,oe,ruiz_mateos,lv_lv,pte_uc,par,aic,cg,psg_eg,av_mec,verde,acn,csd,png_pg,exu,pore,igc,ped,pam,grm,una,ua_cha,ub,ceu,arde,sev,mv,pancal,coa,pee_lv,r_x_c,tagoror,prb,pnem,ppr,lki_emk,pce_ml,cd,un,pnv,pte,psoe_h,ort,erfn,mc_oic,bnpg,c_upc,pg_pog_psg,bean,pcarl,pct,ufv,fjonsa,upn,oce_br,psar___psda,plib,urv,pnpv,unai,pdpcan,smim,psin,ule,e_catala,cic,p_cant,dscc,pprov,lc,ocebr_uce,cna,pippa,poc,cime,fe_uf,uf_fi_at,verdes,ea_eue,le,arm_ade,pap,upca,ua,gil,pgc,pln,e_u,ummp,pec_verde,nps,p_riojano,cng,ag,prex,sees,arcoiris,app,pvpa,mce,atf,aigranc,fpe,aa,fe_a,unida,udla,os,conceyu,piap,abe,pitrcg,if,p_tagoror,mces,us,ince,freixes,ccl,ap,psp_us,pdpc,fdc_edc,udc_idcc,ec_fed,fdi,asdci,aet,an18,rse,fut,caic,esb,pspv,indep,dcv,psg,fjons,afn,indep1,ura,lliga,uan,anepa_cp,fl,div,cja,cups,ccia,uab,uras,fni,ppcan,ms,asa,mfa,anv,indep2,indep3,indep4,dcar,udib,indep5,cup,pscan,faa,pim,indep6,prsv,icu,psde,indep7,indep8,pdg,indep9,indep10,indep11,aeicyu,cuir,cipye,pli,adc,bai,indep12,cai,ic_v,bloc_verds,psm_en,uc_cds,lv_cm,fe,posi,ev_av,pil,es2000,pade,um_inme,ia,blm,fei_fe_2000,pcl,autonomo,ec,pkd,el_papi,a,prex_crex,dg,upz,cnc,peda,li_lit_ci,paviel,ir_pre,usi,pnca,inar,up_can,ural,ccs,csd_l,pf,ppcr,upb,ii,unib,prib,tpc,cives,mash,pdep,an,eh,iu_lv,amaiur,esquerra,equo,cc_nc_pnc,compromis_q,fac,pxc,anticapitalistas,piratacat,espana_2000,rps,hartosorg,pirata,anc,cdl,pcas,uxv,prao,caballas,cxxi,unidad_del_pueblo,andecha,acima,ucit,dec,cyd,prde,internet,pre_r,basta_ya,pdyc,hd,sxt,dne,cc_pnc,ca,na_bai,nc_ccn,aralar,bloc_idpv_ev_ee,unitat_per_les_illes,psd,cenb,ev_ae,pdeal,rcn_nok,ev_lv,aes,e_2000,rc,cva,ei,tc,amd,pdlpea,partido_riojano,abla,extremadura_unida,pxcat,pnf_no_fumadores,uleg,onv,frente,cdes,plci,pleve,li_litci,plrv,centristas_pctr,pcua,ucpic,irv,mupc,puede,piib,pb_ub,ctc,ud_ca,imc,paie,td,cdas,ave,ucl,n_som,axb,ixc,icbur,agruci,phache,ap_pdp_pl,muc,prd,ac_inc,av,upr,pspa,cun,psdc,adei,cl,loc,cn,pmcn,prm,prov,ae_aic,paec,lki,podemos_iu,mas_pais_equo,cup_pr,podemos_eu,cca_pnc_nc,mes_compromis,teruel_existe!,mas_pais,m_pais_cha_equo,mes_esquerra,xav,contigo,pdsje,mas,unidos_si_acps_def,mdyc,aunacv,canda,ecp,podemos_compromis_eupv,cdc,podemos_en_marea_anova_eu,recortes_cero_grupo_verde,bng_nos,si,centro_moderado,ccd_ci,and,jxc,repo,independientes_fia,entaban,alcd,fme,hrts_ln,bloc_ev,"psm_en,eu,ev,er",aralar_zutik,lv_e,lvcm,pcannabis,lvrm,msr,ides,apca,gve,ci,adn,ptpre,up

In [75]:
party_bool = (df_municipio[party_columns_municipio].sum() == 0)
drop_parties = list(party_bool[party_bool].index)
df_municipio_filtered = df_municipio.drop(columns=drop_parties)
df_party_series = df_municipio_filtered.melt(id_vars=['year', "month"], 
                                             value_vars=[x for x in party_columns_municipio if x not in drop_parties])
df_party_series = df_party_series.rename(columns={"variable":"partido", "value":"votos"})
df_party_series = df_party_series[~((df_party_series.year == 2019) & (df_party_series.month == 4))]

top_parties = list(df_party_series.groupby("partido")
                     .sum("votos").sort_values("votos", ascending=False)[:15]
                     .reset_index()["partido"].unique())
df_party_series = df_party_series[df_party_series["partido"].isin(top_parties)]
df_party_series = df_party_series.sort_values(["year", "month", "partido"], ascending=True)
df_party_series["partido"] = df_party_series["partido"].replace({'ap_pdp_pl': "ap", 'ap_pdp': "ap",
                                                            "podemos_iu":"podemos", "podemos_iu_equo":"podemos",
                                                            "iu_lv":"iu"})
df_party_series = df_party_series.groupby(["year","month","partido"]).sum().reset_index()

In [76]:
color_map = {'ucd':"#FFA07A",
             'psoe':"#E52B50", 
             'pp':"#1E90FF", 
             'ap':"#FFFF00",              
             'pce':"#FF0000",              
             'uce':"#FF0000",              
             'ap_pdp':"#FFFF00",              
             'pcoe':"#FF0000",              
             'pst':"#FF0000",              
             'ap_pdp_pl':"#FFFF00",              
             'cds':"#66CDAA",              
             'iu':"#32CD32",
             'fe':"#000000", 
             'ciudadanos':"#FFA500",
             'podemos_iu':"#9370DB",    
             'podemos_iu_equo':"#9370DB",                 
             'podemos':"#9370DB",
             'vox':"#00FF00"}

In [77]:
## GRAFICO --------------------
fig_town_lines = px.line(df_party_series, x="year", y="votos", color="partido", line_group="partido", hover_name="partido",
        line_shape="linear", render_mode="svg", color_discrete_map=color_map,range_x=[1976,2020],template="plotly_white",
        title= f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}")
fig_town_lines.show()